In [34]:
import pandas as pd
import numpy as np
import gzip
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [35]:
metadata_path = "CRyPTIC_reuse_table_20240917.csv"

df = pd.read_csv(metadata_path)

print("Total samples:", len(df))
df.head()

Total samples: 12287


,ENA_RUN,UNIQUEID,AMI_BINARY_PHENOTYPE,BDQ_BINARY_PHENOTYPE,CFZ_BINARY_PHENOTYPE,DLM_BINARY_PHENOTYPE,EMB_BINARY_PHENOTYPE,ETH_BINARY_PHENOTYPE,INH_BINARY_PHENOTYPE,KAN_BINARY_PHENOTYPE,...,INH_PHENOTYPE_QUALITY,KAN_PHENOTYPE_QUALITY,LEV_PHENOTYPE_QUALITY,LZD_PHENOTYPE_QUALITY,MXF_PHENOTYPE_QUALITY,RIF_PHENOTYPE_QUALITY,RFB_PHENOTYPE_QUALITY,ENA_SAMPLE,VCF,REGENOTYPED_VCF
0,ERR4810489,site.02.subj.0001.lab.2014222001.iso.1,S,NaN,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298516,../reproducibility/00/01/08/61/10861/site.02.i...,../reproducibility/00/01/08/61/10861/site.02.i...
1,ERR4810491,site.02.subj.0002.lab.2014222005.iso.1,S,S,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,LOW,HIGH,HIGH,ERS5298518,../reproducibility/00/01/08/63/10863/site.02.i...,../reproducibility/00/01/08/63/10863/site.02.i...
2,ERR4810493,site.02.subj.0004.lab.2014222010.iso.1,S,S,S,NaN,S,I,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298520,../reproducibility/00/01/08/67/10867/site.02.i...,../reproducibility/00/01/08/67/10867/site.02.i...
3,ERR4810494,site.02.subj.0005.lab.2014222011.iso.1,S,S,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298521,../reproducibility/00/01/08/68/10868/site.02.i...,../reproducibility/00/01/08/68/10868/site.02.i...
4,ERR4810495,site.02.subj.0006.lab.2014222013.iso.1,S,S,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298522,../reproducibility/00/01/08/69/10869/site.02.i...,../reproducibility/00/01/08/69/10869/site.02.i...


In [36]:
def convert_label(x):
    if x in (1, "1"):
        return 1
    if x in (0, "0"):
        return 0

    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()
    if x == "R":
        return 1
    elif x == "S":
        return 0
    else:
        return np.nan

In [37]:
drug_columns = [
    "AMI_BINARY_PHENOTYPE",
    "BDQ_BINARY_PHENOTYPE",
    "CFZ_BINARY_PHENOTYPE",
    "DLM_BINARY_PHENOTYPE",
    "EMB_BINARY_PHENOTYPE",
    "ETH_BINARY_PHENOTYPE",
    "INH_BINARY_PHENOTYPE",
    "KAN_BINARY_PHENOTYPE",
    "LEV_BINARY_PHENOTYPE",
    "LZD_BINARY_PHENOTYPE",
    "MXF_BINARY_PHENOTYPE",
    "RFB_BINARY_PHENOTYPE",
    "RIF_BINARY_PHENOTYPE",
]

for col in drug_columns:
    df[col] = df[col].apply(convert_label)

df = df.dropna(subset=drug_columns)

print("Samples after filtering:", len(df))

Samples after filtering: 8749


In [38]:
VCF_DIR = "vcf(2024)"

def extract_variants(vcf_path):
    variants = set()

    with gzip.open(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue

            parts = line.strip().split("\t")
            if len(parts) < 8:
                continue

            pos = int(parts[1])
            ref = parts[3]
            alt_field = parts[4]
            filt = parts[6]

            # Keep only high-confidence calls.
            if filt not in ("PASS", "."):
                continue

            for alt in alt_field.split(","):
                alt = alt.strip()
                if alt:
                    variants.add((pos, ref, alt))

    return variants

In [39]:
sample_mutations = {}

for idx, row in tqdm(df.iterrows(), total=len(df)):
    sample = row["ENA_SAMPLE"]
    vcf_path = os.path.join(VCF_DIR, f"{sample}.vcf.gz")

    if not os.path.exists(vcf_path):
        continue

    variants = extract_variants(vcf_path)
    sample_mutations[sample] = variants

print("Samples processed:", len(sample_mutations))

100%|██████████| 8749/8749 [00:13<00:00, 658.91it/s]

Samples processed: 8749


In [40]:
GENOME_SIZE = 4411532   # MTB genome
WINDOW_SIZE = 100       # Try 50 / 100 / 200

# VCF positions are 1-based, so windows are computed with (pos-1)//WINDOW_SIZE.
NUM_WINDOWS = (GENOME_SIZE - 1) // WINDOW_SIZE + 1

print("Window size:", WINDOW_SIZE)
print("Total windows:", NUM_WINDOWS)

Window size: 100
Total windows: 44116


In [41]:
mutation_counts = {}

for muts in sample_mutations.values():
    for variant in muts:
        mutation_counts[variant] = mutation_counts.get(variant, 0) + 1

print("Total variants before filtering:", len(mutation_counts))

# Drop very-rare variants to reduce noise in window counts.
MIN_COUNT = 2
filtered_mutations = {v for v, c in mutation_counts.items() if c >= MIN_COUNT}

print(f"Variants kept with count >= {MIN_COUNT}: {len(filtered_mutations)}")

Total variants before filtering: 643599
Variants kept with count >= 2: 165134


In [42]:
matrix = []
samples = []

for sample, variants in tqdm(sample_mutations.items()):
    window_vector = np.zeros(NUM_WINDOWS, dtype=np.float32)

    for pos, ref, alt in variants:
        # Use only variants retained by frequency filtering.
        if (pos, ref, alt) not in filtered_mutations:
            continue
        idx = (pos - 1) // WINDOW_SIZE
        if 0 <= idx < NUM_WINDOWS:
            window_vector[idx] += 1

    # TB-DROP style normalization
    window_vector = np.log1p(window_vector)

    matrix.append(window_vector)
    samples.append(sample)

X = pd.DataFrame(matrix, index=samples)

print("Feature matrix shape:", X.shape)

100%|██████████| 8749/8749 [00:05<00:00, 1714.61it/s]


Feature matrix shape: (8749, 44116)


In [43]:
labels = df.set_index("ENA_SAMPLE")[drug_columns]
labels = labels.loc[X.index]

print("Labels shape:", labels.shape)

Labels shape: (8749, 13)


In [44]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Split: 70% train, 15% validation, 15% test (multilabel stratified)
y_all_int = labels.values.astype(np.int64)

msss_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42,
    )
train_idx, temp_idx = next(msss_1.split(X.values, y_all_int))

X_train = X.iloc[train_idx].copy()
y_train = labels.iloc[train_idx].copy()
X_temp = X.iloc[temp_idx].copy()
y_temp = labels.iloc[temp_idx].copy()

msss_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42,
    )
temp_y_int = y_temp.values.astype(np.int64)
val_rel_idx, test_rel_idx = next(msss_2.split(X_temp.values, temp_y_int))

X_val = X_temp.iloc[val_rel_idx].copy()
y_val = y_temp.iloc[val_rel_idx].copy()
X_test = X_temp.iloc[test_rel_idx].copy()
y_test = y_temp.iloc[test_rel_idx].copy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (6124, 44116)
Validation: (1312, 44116)
Test: (1313, 44116)


In [45]:
import os
import subprocess
import torch
import torch.nn as nn
from dataclasses import dataclass

def select_device(preferred_gpu=None):
    if not torch.cuda.is_available():
        return torch.device("cpu")

    # Allow explicit GPU override, e.g. export WDNN_GPU_INDEX=1
    if preferred_gpu is None:
        env_gpu = os.getenv("WDNN_GPU_INDEX")
        if env_gpu is not None and env_gpu.isdigit():
            preferred_gpu = int(env_gpu)

    gpu_count = torch.cuda.device_count()
    if preferred_gpu is not None and 0 <= preferred_gpu < gpu_count:
        return torch.device(f"cuda:{preferred_gpu}")

    try:
        cmd = [
            "nvidia-smi",
            "--query-gpu=index,memory.free",
            "--format=csv,noheader,nounits",
        ]
        output = subprocess.check_output(cmd, text=True).strip().splitlines()
        free_by_gpu = []
        for line in output:
            idx_str, free_str = [x.strip() for x in line.split(",")[:2]]
            free_by_gpu.append((int(idx_str), int(free_str)))
        if free_by_gpu:
            best_gpu = max(free_by_gpu, key=lambda x: x[1])[0]
            return torch.device(f"cuda:{best_gpu}")
    except Exception:
        pass

    return torch.device("cuda:0")

device = select_device()
print("Using device:", device)

@dataclass(frozen=True)
class WDNNConfig:
    input_dim: int = 86392
    hidden_dim: int = 1000
    output_dim: int = 13
    dropout_rate: float = 0.3

    # Matches the extracted SavedModel settings
    bn_epsilon: float = 1e-3
    bn_momentum_keras: float = 0.99

    # Regularizer coefficients from extracted architecture
    kernel_l1: float = 1e-4
    hidden_bias_l2: float = 1e-3
    output_bias_l2: float = 1e-1

class ExactWDNN(nn.Module):
    def __init__(self, config: WDNNConfig) -> None:
        super().__init__()
        self.config = config

        # PyTorch BN momentum uses update weight; Keras uses decay.
        bn_momentum_torch = 1.0 - self.config.bn_momentum_keras

        self.dense1 = nn.Linear(self.config.input_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout = nn.Dropout(p=self.config.dropout_rate)

        self.dense2 = nn.Linear(self.config.hidden_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization_1 = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout_1 = nn.Dropout(p=self.config.dropout_rate)

        self.dense3 = nn.Linear(self.config.hidden_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization_2 = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout_2 = nn.Dropout(p=self.config.dropout_rate)

        self.wdnn_final_layer = nn.Linear(self.config.hidden_dim, self.config.output_dim, bias=True)
        self.output_activation = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dense1(x)
        x = torch.relu(x)
        x = self.batch_normalization(x)
        x = self.dropout(x)

        x = self.dense2(x)
        x = torch.relu(x)
        x = self.batch_normalization_1(x)
        x = self.dropout_1(x)

        x = self.dense3(x)
        x = torch.relu(x)
        x = self.batch_normalization_2(x)
        x = self.dropout_2(x)

        x = self.wdnn_final_layer(x)
        x = self.output_activation(x)
        return x

    def regularization_loss(self) -> torch.Tensor:
        loss = torch.zeros((), dtype=self.dense1.weight.dtype, device=self.dense1.weight.device)

        dense_layers = [self.dense1, self.dense2, self.dense3]
        for layer in dense_layers:
            loss = loss + self.config.kernel_l1 * layer.weight.abs().sum()
            loss = loss + self.config.hidden_bias_l2 * layer.bias.pow(2).sum()

        loss = loss + self.config.kernel_l1 * self.wdnn_final_layer.weight.abs().sum()
        loss = loss + self.config.output_bias_l2 * self.wdnn_final_layer.bias.pow(2).sum()
        return loss

# Build model from current notebook data dimensions
config = WDNNConfig(
    input_dim=X_train.shape[1],
    output_dim=y_train.shape[1],
)

model = ExactWDNN(config).to(device)
print(model)

Using device: cuda:1
ExactWDNN(
  (dense1): Linear(in_features=44116, out_features=1000, bias=True)
  (batch_normalization): BatchNorm1d(1000, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (dense2): Linear(in_features=1000, out_features=1000, bias=True)
  (batch_normalization_1): BatchNorm1d(1000, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (dropout_1): Dropout(p=0.3, inplace=False)
  (dense3): Linear(in_features=1000, out_features=1000, bias=True)
  (batch_normalization_2): BatchNorm1d(1000, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (dropout_2): Dropout(p=0.3, inplace=False)
  (wdnn_final_layer): Linear(in_features=1000, out_features=13, bias=True)
  (output_activation): Sigmoid()
)


In [55]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import copy
import os

# 10-fold multilabel stratified shuffle split settings
N_FOLDS = 10
VAL_SIZE = 0.20
RANDOM_STATE = 42
EPOCHS = 20
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
THRESHOLD = 0.5

SAVE_FOLD_CHECKPOINTS = True
CV_CHECKPOINT_DIR = "checkpoints_cv"
if SAVE_FOLD_CHECKPOINTS:
    os.makedirs(CV_CHECKPOINT_DIR, exist_ok=True)

if "X_train" not in globals() or "X_val" not in globals() or "y_train" not in globals() or "y_val" not in globals():
    raise RuntimeError("X_train/X_val/y_train/y_val not found. Run Cell 11 first.")
if "X_test" not in globals() or "y_test" not in globals():
    raise RuntimeError("X_test/y_test not found. Run Cell 11 first.")

# Keep test holdout untouched. CV is done only on the development split (train + val).
X_dev = pd.concat([X_train, X_val], axis=0)
y_dev = pd.concat([y_train, y_val], axis=0)

common_idx = X_dev.index.intersection(y_dev.index)
if len(common_idx) == 0:
    raise RuntimeError("No overlapping sample IDs between development features and labels.")

X_dev = X_dev.loc[common_idx]
y_dev = y_dev.loc[common_idx, drug_columns]
feature_columns = list(X_dev.columns)

print("Development split shape:", X_dev.shape)
print("Holdout test split shape:", X_test.shape)

X_np = X_dev.values.astype(np.float32)
y_np = y_dev.values.astype(np.float32)
y_np_int = y_np.astype(np.int64)

def safe_div(num, den):
    return float(num / den) if den > 0 else np.nan

def tune_thresholds_per_drug(y_true, y_prob, drug_cols):
    thresholds = np.linspace(0.05, 0.95, 19)
    best_thresholds = {}

    for i, drug in enumerate(drug_cols):
        yt = y_true[:, i].astype(np.int64)
        yp_prob = y_prob[:, i].astype(np.float64)

        # If only one class is present, threshold tuning is not meaningful.
        if len(np.unique(yt)) < 2:
            best_thresholds[drug] = THRESHOLD
            continue

        best_thr = THRESHOLD
        best_f1 = -1.0
        best_rec = -1.0

        for thr in thresholds:
            yp = (yp_prob >= thr).astype(np.int64)
            tp = int(((yp == 1) & (yt == 1)).sum())
            fp = int(((yp == 1) & (yt == 0)).sum())
            fn = int(((yp == 0) & (yt == 1)).sum())

            precision = safe_div(tp, tp + fp)
            recall = safe_div(tp, tp + fn)
            if np.isnan(precision):
                precision = 0.0
            if np.isnan(recall):
                recall = 0.0

            f1 = safe_div(2 * precision * recall, precision + recall)
            if np.isnan(f1):
                f1 = 0.0

            # Tie-break by higher recall so sensitivity does not collapse to zero.
            if (f1 > best_f1) or (np.isclose(f1, best_f1) and recall > best_rec):
                best_f1 = f1
                best_rec = recall
                best_thr = float(thr)

        best_thresholds[drug] = best_thr

    return best_thresholds

def compute_multilabel_metrics(y_true, y_prob, threshold=0.5):
    per_drug = []

    for i, drug in enumerate(drug_columns):
        if isinstance(threshold, dict):
            drug_threshold = float(threshold.get(drug, 0.5))
        else:
            drug_threshold = float(threshold)

        yt = y_true[:, i].astype(np.int64)
        yp_prob = y_prob[:, i].astype(np.float64)
        yp = (yp_prob >= drug_threshold).astype(np.int64)

        tp = int(((yp == 1) & (yt == 1)).sum())
        tn = int(((yp == 0) & (yt == 0)).sum())
        fp = int(((yp == 1) & (yt == 0)).sum())
        fn = int(((yp == 0) & (yt == 1)).sum())

        sensitivity = safe_div(tp, tp + fn)
        specificity = safe_div(tn, tn + fp)
        precision = safe_div(tp, tp + fp)
        npv = safe_div(tn, tn + fn)

        try:
            auc = roc_auc_score(yt, yp_prob)
        except ValueError:
            auc = np.nan

        per_drug.append({
            "drug": drug,
            "threshold": drug_threshold,
            "predicted_positive": int((yp == 1).sum()),
            "actual_positive": int((yt == 1).sum()),
            "AUC": auc,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "NPV": npv,
        })

    per_drug_df = pd.DataFrame(per_drug)
    macro_metrics = {
        "AUC": float(np.nanmean(per_drug_df["AUC"])),
        "sensitivity": float(np.nanmean(per_drug_df["sensitivity"])),
        "specificity": float(np.nanmean(per_drug_df["specificity"])),
        "precision": float(np.nanmean(per_drug_df["precision"])),
        "NPV": float(np.nanmean(per_drug_df["NPV"])),
    }
    return per_drug_df, macro_metrics

msss = MultilabelStratifiedShuffleSplit(
    n_splits=N_FOLDS,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    )

fold_macro_rows = []
fold_per_drug_rows = []

for fold, (train_idx, val_idx) in enumerate(msss.split(X_np, y_np_int), start=1):
    print(f"\n===== Fold {fold}/{N_FOLDS} =====")

    X_train_fold = torch.tensor(X_np[train_idx], dtype=torch.float32)
    y_train_fold = torch.tensor(y_np[train_idx], dtype=torch.float32)
    X_val_fold = torch.tensor(X_np[val_idx], dtype=torch.float32)
    y_val_fold = torch.tensor(y_np[val_idx], dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_train_fold, y_train_fold),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(X_val_fold, y_val_fold),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    fold_config = WDNNConfig(
        input_dim=X_np.shape[1],
        output_dim=y_np.shape[1],
    )
    model = ExactWDNN(fold_config).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_loss = float("inf")
    best_epoch = 0
    best_state_dict = None
    best_macro = None
    best_per_drug = None
    best_thresholds = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb) + model.regularization_loss()
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * xb.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)

        model.eval()
        val_loss_sum = 0.0
        val_probs_list = []
        val_targets_list = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb) + model.regularization_loss()
                val_loss_sum += loss.item() * xb.size(0)
                val_probs_list.append(preds.detach().cpu().numpy())
                val_targets_list.append(yb.detach().cpu().numpy())

        val_loss = val_loss_sum / len(val_loader.dataset)
        val_probs = np.vstack(val_probs_list)
        val_targets = np.vstack(val_targets_list)

        epoch_thresholds = tune_thresholds_per_drug(
            y_true=val_targets,
            y_prob=val_probs,
            drug_cols=drug_columns,
        )
        per_drug_df, macro = compute_multilabel_metrics(
            y_true=val_targets,
            y_prob=val_probs,
            threshold=epoch_thresholds,
        )

        print(
            f"Fold {fold} Epoch {epoch}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | "
            f"AUC: {macro['AUC']:.4f} | Sens: {macro['sensitivity']:.4f} | "
            f"Spec: {macro['specificity']:.4f} | Prec: {macro['precision']:.4f} | NPV: {macro['NPV']:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            best_macro = macro
            best_per_drug = per_drug_df.copy()
            best_thresholds = dict(epoch_thresholds)

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    if SAVE_FOLD_CHECKPOINTS and best_state_dict is not None:
        fold_ckpt_path = os.path.join(CV_CHECKPOINT_DIR, f"wdnn_fold_{fold:02d}_best.pt")
        torch.save(
            {
                "fold": fold,
                "best_epoch": best_epoch,
                "model_state_dict": best_state_dict,
                "config": fold_config,
                "best_val_loss": best_val_loss,
                "best_macro_metrics": best_macro,
                "best_thresholds": best_thresholds,
                "feature_columns": feature_columns,
                "drug_columns": list(drug_columns),
            },
            fold_ckpt_path,
        )
        print(f"Saved fold {fold} best checkpoint: {fold_ckpt_path}")

    fold_row = {
        "fold": fold,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "AUC": best_macro["AUC"],
        "sensitivity": best_macro["sensitivity"],
        "specificity": best_macro["specificity"],
        "precision": best_macro["precision"],
        "NPV": best_macro["NPV"],
    }
    fold_macro_rows.append(fold_row)

    best_per_drug = best_per_drug.copy()
    best_per_drug.insert(0, "fold", fold)
    best_per_drug.insert(1, "best_epoch", best_epoch)
    fold_per_drug_rows.append(best_per_drug)

fold_macro_df = pd.DataFrame(fold_macro_rows)
fold_per_drug_df = pd.concat(fold_per_drug_rows, ignore_index=True)

print("\n===== 10-Fold Macro Summary on Development Split (best epoch per fold) =====")
display(fold_macro_df)

summary_mean = fold_macro_df[["AUC", "sensitivity", "specificity", "precision", "NPV"]].mean()
summary_std = fold_macro_df[["AUC", "sensitivity", "specificity", "precision", "NPV"]].std()
summary_df = pd.DataFrame({"mean": summary_mean, "std": summary_std})

print("\n===== Overall Mean +/- Std Across Folds =====")
display(summary_df)

print("\n===== Per-Drug Metrics Across Folds (best epoch per fold) =====")
display(fold_per_drug_df)

Development split shape: (7436, 44116)
Holdout test split shape: (1313, 44116)

===== Fold 1/10 =====
Fold 1 Epoch 1/20 | Train loss: 8.6334 | Val loss: 5.7478 | AUC: 0.6380 | Sens: 0.6845 | Spec: 0.5219 | Prec: 0.3209 | NPV: 0.8966
Fold 1 Epoch 2/20 | Train loss: 4.7622 | Val loss: 3.7645 | AUC: 0.6954 | Sens: 0.5425 | Spec: 0.6916 | Prec: 0.4141 | NPV: 0.9079
Fold 1 Epoch 3/20 | Train loss: 3.1456 | Val loss: 2.6959 | AUC: 0.7553 | Sens: 0.5444 | Spec: 0.7276 | Prec: 0.4704 | NPV: 0.8852
Fold 1 Epoch 4/20 | Train loss: 2.3041 | Val loss: 2.0689 | AUC: 0.7990 | Sens: 0.5620 | Spec: 0.7203 | Prec: 0.4324 | NPV: 0.9230
Fold 1 Epoch 5/20 | Train loss: 1.8603 | Val loss: 1.7608 | AUC: 0.7470 | Sens: 0.6059 | Spec: 0.6117 | Prec: 0.4120 | NPV: 0.8928
Fold 1 Epoch 6/20 | Train loss: 1.6258 | Val loss: 1.5830 | AUC: 0.5730 | Sens: 0.7912 | Spec: 0.3532 | Prec: 0.3460 | NPV: 0.8866
Fold 1 Epoch 7/20 | Train loss: 1.5164 | Val loss: 1.5072 | AUC: 0.6920 | Sens: 0.6886 | Spec: 0.5824 | Prec: 0.

,fold,best_epoch,best_val_loss,AUC,sensitivity,specificity,precision,NPV
0,1,18,1.277461,0.824979,0.417497,0.934018,0.706198,0.950534
1,2,19,1.187747,0.823979,0.469771,0.911541,0.640545,0.948318
2,3,19,1.270744,0.794703,0.432844,0.849846,0.582475,0.929157
3,4,16,1.268200,0.795161,0.466168,0.783086,0.473980,0.933956
4,5,19,1.245403,0.815721,0.457904,0.930880,0.592657,0.951535
5,6,18,1.317332,0.813141,0.465274,0.743539,0.342640,0.965307
6,7,20,1.288687,0.836224,0.417426,0.951235,0.689823,0.944526
7,8,17,1.315360,0.818749,0.492330,0.761330,0.471707,0.939894
8,9,20,1.209526,0.792317,0.522768,0.897958,0.432336,0.962014
9,10,14,1.303885,0.817331,0.468532,0.743793,0.530009,0.948271



===== Overall Mean +/- Std Across Folds =====


,mean,std
AUC,0.813230,0.014693
sensitivity,0.461051,0.032607
specificity,0.850723,0.084909
precision,0.546237,0.117393
NPV,0.947351,0.011259



===== Per-Drug Metrics Across Folds (best epoch per fold) =====


,fold,best_epoch,drug,threshold,predicted_positive,actual_positive,AUC,sensitivity,specificity,precision,NPV
0,1,18,AMI_BINARY_PHENOTYPE,0.05,9,77,0.882638,0.103896,0.999291,0.888889,0.953347
1,1,18,BDQ_BINARY_PHENOTYPE,0.05,0,11,0.777067,0.000000,1.000000,NaN,0.992608
2,1,18,CFZ_BINARY_PHENOTYPE,0.05,14,62,0.649459,0.129032,0.995792,0.571429,0.963365
3,1,18,DLM_BINARY_PHENOTYPE,0.05,0,20,0.595572,0.000000,1.000000,NaN,0.986559
4,1,18,EMB_BINARY_PHENOTYPE,0.10,317,300,0.906142,0.776667,0.929293,0.735016,0.942784
...,...,...,...,...,...,...,...,...,...,...,...
125,10,14,LEV_BINARY_PHENOTYPE,0.05,1423,202,0.830274,0.975248,0.046656,0.138440,0.923077
126,10,14,LZD_BINARY_PHENOTYPE,0.05,0,16,0.743249,0.000000,1.000000,NaN,0.989247
127,10,14,MXF_BINARY_PHENOTYPE,0.05,352,161,0.855294,0.795031,0.831198,0.363636,0.970951
128,10,14,RFB_BINARY_PHENOTYPE,0.20,219,418,0.941504,0.483254,0.984112,0.922374,0.829787


In [56]:
# Evaluate best saved fold on holdout test split (X_test, y_test)
import os

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

# Reuse selected training device when available; otherwise choose best available GPU.
device = globals().get("device", torch.device("cuda:0" if torch.cuda.is_available() else "cpu"))
print("Using device:", device)

THRESHOLD = 0.5
CV_CHECKPOINT_DIR = "checkpoints_cv"

def safe_div(num, den):
    return float(num / den) if den > 0 else np.nan

def compute_multilabel_metrics(y_true, y_prob, drug_cols, threshold=0.5):
    per_drug = []

    for i, drug in enumerate(drug_cols):
        if isinstance(threshold, dict):
            drug_threshold = float(threshold.get(drug, 0.5))
        else:
            drug_threshold = float(threshold)

        yt = y_true[:, i].astype(np.int64)
        yp_prob = y_prob[:, i].astype(np.float64)
        yp = (yp_prob >= drug_threshold).astype(np.int64)

        tp = int(((yp == 1) & (yt == 1)).sum())
        tn = int(((yp == 0) & (yt == 0)).sum())
        fp = int(((yp == 1) & (yt == 0)).sum())
        fn = int(((yp == 0) & (yt == 1)).sum())

        sensitivity = safe_div(tp, tp + fn)
        specificity = safe_div(tn, tn + fp)
        precision = safe_div(tp, tp + fp)
        npv = safe_div(tn, tn + fn)

        try:
            auc = roc_auc_score(yt, yp_prob)
        except ValueError:
            auc = np.nan

        per_drug.append({
            "drug": drug,
            "threshold": drug_threshold,
            "predicted_positive": int((yp == 1).sum()),
            "actual_positive": int((yt == 1).sum()),
            "AUC": auc,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "NPV": npv,
        })

    per_drug_df = pd.DataFrame(per_drug)
    macro_metrics = {
        "AUC": float(np.nanmean(per_drug_df["AUC"])),
        "sensitivity": float(np.nanmean(per_drug_df["sensitivity"])),
        "specificity": float(np.nanmean(per_drug_df["specificity"])),
        "precision": float(np.nanmean(per_drug_df["precision"])),
        "NPV": float(np.nanmean(per_drug_df["NPV"])),
    }
    return per_drug_df, macro_metrics

def load_checkpoint(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

if not os.path.isdir(CV_CHECKPOINT_DIR):
    raise FileNotFoundError(f"Checkpoint directory not found: {CV_CHECKPOINT_DIR}")

if "drug_columns" not in globals():
    raise RuntimeError("drug_columns not found. Run Cell 4 first.")
if "X_test" not in globals() or "y_test" not in globals():
    raise RuntimeError("X_test/y_test not found. Run Cell 11 first.")

test_drug_columns = list(drug_columns)
missing_cols = [c for c in test_drug_columns if c not in y_test.columns]
if missing_cols:
    raise ValueError(f"Missing drug columns in y_test: {missing_cols}")

common_idx = X_test.index.intersection(y_test.index)
if len(common_idx) == 0:
    raise RuntimeError("No overlapping sample IDs between X_test and y_test.")

X_test_eval = X_test.loc[common_idx]
y_test_eval = y_test.loc[common_idx, test_drug_columns]

checkpoint_rows = []
skipped_incompatible = []
for fname in sorted(os.listdir(CV_CHECKPOINT_DIR)):
    if not (fname.startswith("wdnn_fold_") and fname.endswith("_best.pt")):
        continue
    ckpt_path = os.path.join(CV_CHECKPOINT_DIR, fname)
    ckpt = load_checkpoint(ckpt_path, map_location="cpu")
    fold_value = ckpt.get("fold")
    if fold_value is None:
        continue

    ckpt_feature_cols = ckpt.get("feature_columns")
    if ckpt_feature_cols is not None:
        ckpt_feature_cols = list(ckpt_feature_cols)
        missing_feature_cols = [c for c in ckpt_feature_cols if c not in X_test_eval.columns]
        if missing_feature_cols:
            skipped_incompatible.append({
                "path": ckpt_path,
                "fold": fold_value,
                "reason": f"Missing feature columns ({len(missing_feature_cols)})",
            })
            continue
        X_test_candidate = X_test_eval.loc[:, ckpt_feature_cols]
    else:
        X_test_candidate = X_test_eval

    state = ckpt.get("model_state_dict", {})
    dense1_w = state.get("dense1.weight")
    final_w = state.get("wdnn_final_layer.weight")
    ckpt_in_dim = int(dense1_w.shape[1]) if dense1_w is not None else None
    ckpt_out_dim = int(final_w.shape[0]) if final_w is not None else None

    expected_input_dim = int(X_test_candidate.shape[1])
    expected_output_dim = int(len(test_drug_columns))
    if ckpt_in_dim != expected_input_dim or ckpt_out_dim != expected_output_dim:
        skipped_incompatible.append({
            "path": ckpt_path,
            "fold": fold_value,
            "reason": "Input/output dimension mismatch",
            "input_dim": ckpt_in_dim,
            "output_dim": ckpt_out_dim,
        })
        continue

    macro = ckpt.get("best_macro_metrics", {}) or {}
    checkpoint_rows.append({
        "fold": int(fold_value),
        "path": ckpt_path,
        "AUC": float(macro.get("AUC", np.nan)),
        "sensitivity": float(macro.get("sensitivity", np.nan)),
        "specificity": float(macro.get("specificity", np.nan)),
        "precision": float(macro.get("precision", np.nan)),
        "NPV": float(macro.get("NPV", np.nan)),
        "feature_columns": ckpt_feature_cols,
    })

if not checkpoint_rows:
    skipped_df = pd.DataFrame(skipped_incompatible)
    print("No compatible checkpoints found for current holdout test shape.")
    if not skipped_df.empty:
        print("\nFound incompatible checkpoints:")
        display(skipped_df)
    raise RuntimeError(
        "No compatible checkpoints. Run Cell 13 to train and save checkpoints for the current feature-engineering setup, then rerun this cell."
    )

fold_macro_df = pd.DataFrame(checkpoint_rows).sort_values(
    by=["AUC", "fold"], ascending=[False, True], na_position="last"
).reset_index(drop=True)

best_fold = int(fold_macro_df.loc[0, "fold"] )
best_fold_ckpt_path = fold_macro_df.loc[0, "path"]
best_fold_feature_cols = fold_macro_df.loc[0, "feature_columns"]
print(f"\nBest fold based on saved checkpoint AUC: Fold {best_fold}")
display(fold_macro_df[["fold", "AUC", "sensitivity", "specificity", "precision", "NPV"]])

checkpoint = load_checkpoint(best_fold_ckpt_path, map_location="cpu")
cfg_obj = checkpoint.get("config")
best_thresholds = checkpoint.get("best_thresholds")
if not isinstance(best_thresholds, dict):
    best_thresholds = THRESHOLD
    print("No saved per-drug thresholds found in checkpoint; using fixed threshold 0.5")
else:
    print("Using saved per-drug thresholds from best fold checkpoint.")

cfg_values = {}
if isinstance(cfg_obj, dict):
    for key in WDNNConfig.__dataclass_fields__.keys():
        if key in cfg_obj:
            cfg_values[key] = cfg_obj[key]
elif cfg_obj is not None:
    for key in WDNNConfig.__dataclass_fields__.keys():
        if hasattr(cfg_obj, key):
            cfg_values[key] = getattr(cfg_obj, key)

if best_fold_feature_cols is not None:
    X_test_final = X_test_eval.loc[:, list(best_fold_feature_cols)]
else:
    X_test_final = X_test_eval

cfg_values["input_dim"] = int(X_test_final.shape[1])
cfg_values["output_dim"] = int(len(test_drug_columns))

base_cfg = WDNNConfig()
best_fold_config = WDNNConfig(
    input_dim=int(cfg_values.get("input_dim", base_cfg.input_dim)),
    hidden_dim=int(cfg_values.get("hidden_dim", base_cfg.hidden_dim)),
    output_dim=int(cfg_values.get("output_dim", base_cfg.output_dim)),
    dropout_rate=float(cfg_values.get("dropout_rate", base_cfg.dropout_rate)),
    bn_epsilon=float(cfg_values.get("bn_epsilon", base_cfg.bn_epsilon)),
    bn_momentum_keras=float(cfg_values.get("bn_momentum_keras", base_cfg.bn_momentum_keras)),
    kernel_l1=float(cfg_values.get("kernel_l1", base_cfg.kernel_l1)),
    hidden_bias_l2=float(cfg_values.get("hidden_bias_l2", base_cfg.hidden_bias_l2)),
    output_bias_l2=float(cfg_values.get("output_bias_l2", base_cfg.output_bias_l2)),
)

# Try selected device first, but automatically fall back to CPU on OOM.
eval_device = device
best_model = ExactWDNN(best_fold_config)
try:
    best_model = best_model.to(eval_device)
except torch.OutOfMemoryError:
    eval_device = torch.device("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    best_model = best_model.to(eval_device)
    print("GPU OOM while loading model. Falling back to CPU evaluation.")

best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

X_test_tensor = torch.tensor(X_test_final.values.astype(np.float32), dtype=torch.float32).to(eval_device)
y_test_tensor = torch.tensor(y_test_eval.values.astype(np.float32), dtype=torch.float32).to(eval_device)

with torch.no_grad():
    test_preds = best_model(X_test_tensor).cpu().numpy()
    test_targets = y_test_tensor.cpu().numpy()

test_per_drug_df, test_macro = compute_multilabel_metrics(
    y_true=test_targets,
    y_prob=test_preds,
    drug_cols=test_drug_columns,
    threshold=best_thresholds,
)

print(f"\n===== Holdout Test Metrics for Best Fold (Fold {best_fold}) =====")
print(
    f"AUC: {test_macro['AUC']:.4f} | Sensitivity: {test_macro['sensitivity']:.4f} | "
    f"Specificity: {test_macro['specificity']:.4f} | Precision: {test_macro['precision']:.4f} | NPV: {test_macro['NPV']:.4f}"
)
print("\nPer-drug metrics on holdout test set:")
display(test_per_drug_df)

Using device: cuda:1

Best fold based on saved checkpoint AUC: Fold 7


,fold,AUC,sensitivity,specificity,precision,NPV
0,7,0.836224,0.417426,0.951235,0.689823,0.944526
1,1,0.824979,0.417497,0.934018,0.706198,0.950534
2,2,0.823979,0.469771,0.911541,0.640545,0.948318
3,8,0.818749,0.492330,0.761330,0.471707,0.939894
4,10,0.817331,0.468532,0.743793,0.530009,0.948271
5,5,0.815721,0.457904,0.930880,0.592657,0.951535
6,6,0.813141,0.465274,0.743539,0.342640,0.965307
7,4,0.795161,0.466168,0.783086,0.473980,0.933956
8,3,0.794703,0.432844,0.849846,0.582475,0.929157
9,9,0.792317,0.522768,0.897958,0.432336,0.962014


Using saved per-drug thresholds from best fold checkpoint.

===== Holdout Test Metrics for Best Fold (Fold 7) =====
AUC: 0.8154 | Sensitivity: 0.4316 | Specificity: 0.9529 | Precision: 0.6860 | NPV: 0.9450

Per-drug metrics on holdout test set:


,drug,threshold,predicted_positive,actual_positive,AUC,sensitivity,specificity,precision,NPV
0,AMI_BINARY_PHENOTYPE,0.05,21,68,0.869537,0.279412,0.998394,0.904762,0.962074
1,BDQ_BINARY_PHENOTYPE,0.05,0,10,0.764620,0.000000,1.000000,NaN,0.992384
2,CFZ_BINARY_PHENOTYPE,0.05,228,55,0.598237,0.254545,0.829889,0.061404,0.962212
3,DLM_BINARY_PHENOTYPE,0.05,0,18,0.409781,0.000000,1.000000,NaN,0.986291
4,EMB_BINARY_PHENOTYPE,0.10,126,265,0.953500,0.449057,0.993321,0.944444,0.877001
5,ETH_BINARY_PHENOTYPE,0.10,119,181,0.881123,0.530387,0.979682,0.806723,0.928811
6,INH_BINARY_PHENOTYPE,0.35,491,521,0.938611,0.831094,0.926768,0.881874,0.892944
7,KAN_BINARY_PHENOTYPE,0.05,152,89,0.846111,0.606742,0.919935,0.355263,0.969854
8,LEV_BINARY_PHENOTYPE,0.10,51,178,0.883562,0.213483,0.988546,0.745098,0.889065
9,LZD_BINARY_PHENOTYPE,0.05,0,14,0.639283,0.000000,1.000000,NaN,0.989337


In [57]:
# Display test results table produced in Cell 16
if "test_per_drug_df" not in globals():
    print("test_per_drug_df not found. Run Cell 15 and then Cell 16 first.")
else:
    test_summary_df = test_per_drug_df.copy()
    print("\n===== Test Set Summary Table =====")
    display(test_summary_df)


===== Test Set Summary Table =====


,drug,threshold,predicted_positive,actual_positive,AUC,sensitivity,specificity,precision,NPV
0,AMI_BINARY_PHENOTYPE,0.05,21,68,0.869537,0.279412,0.998394,0.904762,0.962074
1,BDQ_BINARY_PHENOTYPE,0.05,0,10,0.764620,0.000000,1.000000,NaN,0.992384
2,CFZ_BINARY_PHENOTYPE,0.05,228,55,0.598237,0.254545,0.829889,0.061404,0.962212
3,DLM_BINARY_PHENOTYPE,0.05,0,18,0.409781,0.000000,1.000000,NaN,0.986291
4,EMB_BINARY_PHENOTYPE,0.10,126,265,0.953500,0.449057,0.993321,0.944444,0.877001
5,ETH_BINARY_PHENOTYPE,0.10,119,181,0.881123,0.530387,0.979682,0.806723,0.928811
6,INH_BINARY_PHENOTYPE,0.35,491,521,0.938611,0.831094,0.926768,0.881874,0.892944
7,KAN_BINARY_PHENOTYPE,0.05,152,89,0.846111,0.606742,0.919935,0.355263,0.969854
8,LEV_BINARY_PHENOTYPE,0.10,51,178,0.883562,0.213483,0.988546,0.745098,0.889065
9,LZD_BINARY_PHENOTYPE,0.05,0,14,0.639283,0.000000,1.000000,NaN,0.989337


In [59]:
# Quick diagnostic for Cell 15 output
if "test_summary_df" not in globals():
    raise RuntimeError("test_summary_df not found. Run Cell 15 first.")

diag_df = test_summary_df.copy()

nan_precision_rows = diag_df[diag_df["precision"].isna()]
zero_sens_rows = diag_df[diag_df["sensitivity"].fillna(np.nan) == 0]

print("Rows in test summary:", len(diag_df))
print("Drugs with precision NaN:", len(nan_precision_rows))
print("Drugs with sensitivity == 0:", len(zero_sens_rows))

if len(nan_precision_rows) > 0:
    print("\nDrugs with precision NaN:")
    display(nan_precision_rows[["drug", "AUC", "sensitivity", "specificity", "precision", "NPV"]])

if len(zero_sens_rows) > 0:
    print("\nDrugs with sensitivity == 0:")
    display(zero_sens_rows[["drug", "AUC", "sensitivity", "specificity", "precision", "NPV"]])

Rows in test summary: 13
Drugs with precision NaN: 3
Drugs with sensitivity == 0: 3

Drugs with precision NaN:


,drug,AUC,sensitivity,specificity,precision,NPV
1,BDQ_BINARY_PHENOTYPE,0.764620,0.0,1.0,NaN,0.992384
3,DLM_BINARY_PHENOTYPE,0.409781,0.0,1.0,NaN,0.986291
9,LZD_BINARY_PHENOTYPE,0.639283,0.0,1.0,NaN,0.989337



Drugs with sensitivity == 0:


,drug,AUC,sensitivity,specificity,precision,NPV
1,BDQ_BINARY_PHENOTYPE,0.764620,0.0,1.0,NaN,0.992384
3,DLM_BINARY_PHENOTYPE,0.409781,0.0,1.0,NaN,0.986291
9,LZD_BINARY_PHENOTYPE,0.639283,0.0,1.0,NaN,0.989337
